In [1]:
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import soundfile as sf
import gradio as gr
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")

# 加载处理器和模型
processor = AutoProcessor.from_pretrained("facebook/musicgen-medium")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-medium").to(device)

I0000 00:00:1778735565.455043  821919 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778735565.522528  821919 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778735571.304809  821919 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Using device: cuda


In [2]:
from transformers import pipeline
import scipy

synthesiser = pipeline("text-to-audio", "facebook/musicgen-small")

music = synthesiser("jazz music with a soothing melody", forward_params={"do_sample": True})

scipy.io.wavfile.write("musicgen_out.wav", rate=music["sampling_rate"], data=music["audio"])


Device set to use cuda:0


In [5]:
import random
from pathlib import Path

def get_random_absolute_paths(directory_path, num_samples=100):
    """
    读取目录下所有文件，随机抽取指定数量，并返回绝对路径列表。
    """
    # 将字符串路径转换为 Path 对象
    path = Path(directory_path)
    
    # 确保目录存在
    if not path.is_dir():
        raise ValueError(f"找不到指定的目录: {directory_path}")

    # 获取该目录下所有文件的绝对路径 (排除文件夹)
    # resolve() 方法会自动将路径转换为绝对路径
    all_files = [p.resolve() for p in path.iterdir() if p.is_file()]

    # 安全检查：如果目录下的文件总数少于要抽取的数量，则返回所有文件
    if len(all_files) < num_samples:
        print(f"⚠️ 提示：目录下仅有 {len(all_files)} 个文件，将返回全部文件。")
        num_samples = len(all_files)

    # 无放回地随机抽取指定数量的文件
    selected_files = random.sample(all_files, num_samples)

    # 将 Path 对象转换回字符串格式的列表
    return [str(f) for f in selected_files]

def inspect_pt_file(file_path):
    """
    读取 .pt 文件并打印其内部结构或 keys
    """
    print(f"\n🔍 正在解析文件: {file_path}")
    
    try:
        # 使用 map_location='cpu' 防止在没有 GPU 的环境下报错
        data = torch.load(file_path, map_location='cpu')
        
        # 1. 如果存的是字典（最常见的情况，比如包含 'audio', 'label' 等）
        if isinstance(data, dict):
            keys = list(data.keys())
            print(f"✅ 这是一个字典格式的数据，包含 {len(keys)} 个 Key:")
            for key in keys:
                # 顺便打印一下每个 key 对应数据的类型，方便你了解数据结构
                val_type = type(data[key])
                if isinstance(data[key], torch.Tensor):
                    print(f"   - '{key}': {val_type.__name__} (Shape: {list(data[key].shape)})")
                else:
                    print(f"   - '{key}': {val_type.__name__}")
            return keys
            
        # 2. 如果存的是纯张量 (Tensor)，它没有 key，只有 shape
        elif isinstance(data, torch.Tensor):
            print(f"ℹ️ 这是一个纯 Tensor 数据 (没有 key)。")
            print(f"   - Shape: {list(data.shape)}")
            print(f"   - Dtype: {data.dtype}")
            return None
            
        # 3. 其他 Python 对象
        else:
            print(f"ℹ️ 这是一个 {type(data).__name__} 类型的数据。")
            return None
            
    except Exception as e:
        print(f"❌ 读取文件时发生错误: {e}")
        return None

# ---------------- 使用示例 ----------------

# 这是从你的截图中提取的终端当前绝对路径
target_dir = "/inspire/qb-ilm/project/powersystem/public/tieruian_public/MTG_dataset/mtg_tokens_mucodec_val_w_MSMlabel"

# 获取 100 个随机文件的绝对路径列表
random_100_paths = get_random_absolute_paths(target_dir, 100)

# 打印前 5 个路径查看效果
print(f"成功获取 {len(random_100_paths)} 个文件的绝对路径，前 5 个如下：")
for p in random_100_paths[:5]:
    print(p)

# 假设 random_100_paths 是上一步获取的列表
if random_100_paths:
    # 我们只挑第一个文件来看看结构
    sample_file = random_100_paths[0]
    inspect_pt_file(sample_file)

成功获取 100 个文件的绝对路径，前 5 个如下：
/inspire/qb-ilm/project/powersystem/public/tieruian_public/MTG_dataset/mtg_tokens_mucodec_val_w_MSMlabel/track_249994.pt
/inspire/qb-ilm/project/powersystem/public/tieruian_public/MTG_dataset/mtg_tokens_mucodec_val_w_MSMlabel/track_387420.pt
/inspire/qb-ilm/project/powersystem/public/tieruian_public/MTG_dataset/mtg_tokens_mucodec_val_w_MSMlabel/track_1151556.pt
/inspire/qb-ilm/project/powersystem/public/tieruian_public/MTG_dataset/mtg_tokens_mucodec_val_w_MSMlabel/track_662085.pt
/inspire/qb-ilm/project/powersystem/public/tieruian_public/MTG_dataset/mtg_tokens_mucodec_val_w_MSMlabel/track_1076450.pt

🔍 正在解析文件: /inspire/qb-ilm/project/powersystem/public/tieruian_public/MTG_dataset/mtg_tokens_mucodec_val_w_MSMlabel/track_249994.pt
✅ 这是一个字典格式的数据，包含 8 个 Key:
   - 'track_id': str
   - 'tokens': list
   - 'text_prompt': str
   - 'text_embeds': Tensor (Shape: [1, 23, 768])
   - 'num_chunks': int
   - 'sample_rate': int
   - 'chunk_size_sec': int
   - 'codec': str


In [6]:
import random
import torch
import scipy.io.wavfile
from pathlib import Path
from transformers import pipeline, set_seed
from tqdm import tqdm

# --- 0. 固定全局随机种子 ---
SEED = 42
random.seed(SEED)        # 固定 Python 内置的随机抽样
set_seed(SEED)           # 固定 Transformers, PyTorch (CPU/GPU) 和 NumPy 的随机种子
print(f"🔒 已固定全局随机种子为: {SEED}")

# --- 1. 路径与参数配置 ---
input_dir = "/inspire/qb-ilm/project/powersystem/public/tieruian_public/MTG_dataset/mtg_tokens_mucodec_val_w_MSMlabel"
output_dir = "/inspire/hdd/project/powersystem/tieruian-253108120115/chenxie/music_project/output/musicgen_medium"
num_samples = 100

Path(output_dir).mkdir(parents=True, exist_ok=True)

# --- 2. 获取随机文件列表 ---
print("🔍 正在扫描并抽取随机文件...")
input_path = Path(input_dir)
all_pt_files = [p.resolve() for p in input_path.iterdir() if p.is_file() and p.suffix == '.pt']

if len(all_pt_files) < num_samples:
    num_samples = len(all_pt_files)
    
# 因为上面设置了 random.seed(42)，这里每次运行抽出来的 100 个文件都会一模一样
selected_files = random.sample(all_pt_files, num_samples)

# --- 3. 初始化 MusicGen 模型 ---
print("⏳ 正在加载 MusicGen 模型到 GPU...")
synthesiser = pipeline("text-to-audio", "facebook/musicgen-small", device=0)

# --- 4. 批量生成音频 ---
print(f"🚀 开始为 {num_samples} 个 prompt 生成音频...")

for pt_file in tqdm(selected_files, desc="生成进度"):
    try:
        data = torch.load(pt_file, map_location='cpu')
        prompt = data.get("text_prompt")
        
        if not prompt:
            continue
            
        if isinstance(prompt, list):
            prompt = prompt[0]
            
        # 因为设置了 set_seed(42)，即使 do_sample=True，生成的音频也是确定的
        music = synthesiser(prompt, forward_params={"do_sample": True})
        
        out_filename = pt_file.stem + ".wav"
        out_filepath = Path(output_dir) / out_filename
        
        scipy.io.wavfile.write(
            out_filepath, 
            rate=music["sampling_rate"], 
            data=music["audio"][0].T  
        )
        
    except Exception as e:
        print(f"\n❌ 处理文件 {pt_file.name} 时发生错误: {e}")

print(f"\n✅ 全部完成！音频已保存至: {output_dir}")


🔒 已固定全局随机种子为: 42
🔍 正在扫描并抽取随机文件...
⏳ 正在加载 MusicGen 模型到 GPU...


Device set to use cuda:0


🚀 开始为 100 个 prompt 生成音频...


生成进度:   6%|▌         | 6/100 [02:16<35:31, 22.67s/it]


KeyboardInterrupt: 

In [ ]:
from datasets import load_dataset

ds = load_dataset("google/MusicCaps")